# Agent loop lab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ankitrgupta1/silicon-to-agents/blob/main/labs/agent-loop.ipynb)

Build the loop that turns a model into an agent: tool definitions, tool calls, results fed back, error handling, an approval gate and a step limit. Companion to [The agent loop](../08-agents/agent-loop.md). Runs on CPU with no API key: simple stand-in models make the decisions a real LLM would.

In [ ]:
try:
    import s2a
except ImportError:
    %pip install -q git+https://github.com/ankitrgupta1/silicon-to-agents

import json, os
from s2a.agent import (Tool, ToolCall, ModelResponse, ScriptedModel, RuleBasedModel, CALCULATOR,
                       run_agent, tokens_processed)
from s2a.rag import Retriever, load_markdown_corpus
from s2a.kv_cache import ModelConfig, kv_cache_bytes, human_bytes

ROOT = ".." if os.path.exists("../03-foundations") else "silicon-to-agents"
if not os.path.exists(ROOT):
    !git clone -q --depth 1 https://github.com/ankitrgupta1/silicon-to-agents

## 1. Tools: a description, a schema, and code

The model only ever sees the name, description and parameter schema. The code stays on our side. We'll give the agent three tools, all built from earlier labs.

In [ ]:
retriever = Retriever(load_markdown_corpus(ROOT))

search_docs = Tool(
    name="search_docs",
    description="Search the silicon-to-agents knowledge base. Returns the 3 most relevant passages with their source pages.",
    parameters={"type": "object", "properties": {"query": {"type": "string"}}, "required": ["query"]},
    fn=lambda query: [f"{c.source}: {c.text[:150]}" for c in retriever.search(query, k=3)],
)

kv_cache_size = Tool(
    name="kv_cache_size",
    description="KV-cache memory for a model shape and context length, in BF16.",
    parameters={"type": "object", "required": ["layers", "kv_heads", "head_dim", "tokens"],
                "properties": {k: {"type": "integer"} for k in ["layers", "kv_heads", "head_dim", "tokens"]}},
    fn=lambda layers, kv_heads, head_dim, tokens: human_bytes(kv_cache_bytes(ModelConfig("m", layers, kv_heads, head_dim), tokens)),
)

tools = [CALCULATOR, search_docs, kv_cache_size]
print(json.dumps([t.schema() for t in tools], indent=1)[:900], "...")

## 2. Run the loop

`RuleBasedModel` stands in for an LLM: it calls the calculator for arithmetic, searches the docs otherwise, then answers from the tool result. Watch the messages go back and forth.

In [ ]:
for question in ["What is 4096 * 14336?", "How does prefix caching save money?"]:
    result = run_agent(RuleBasedModel(), tools, question)
    print(f"{result.steps} model calls, stopped because: {result.stopped_because}")
    print(result.trace(), "\n")

## 3. Several steps, chained

A scripted model shows what a capable LLM does on a harder question: use one tool's result to decide the next call.

In [ ]:
plan = ScriptedModel([
    ModelResponse(text="First find the model's shape.", tool_calls=[ToolCall("search_docs", {"query": "Llama 3 8B layers KV heads head dimension"})]),
    ModelResponse(text="32 layers, 8 KV heads, head dim 128. Now size the cache.",
                  tool_calls=[ToolCall("kv_cache_size", {"layers": 32, "kv_heads": 8, "head_dim": 128, "tokens": 32768})]),
    ModelResponse(text="A 32k-token conversation needs 4 GiB of KV cache for Llama 3 8B in BF16."),
])
result = run_agent(plan, tools, "How much KV cache does Llama 3 8B need for a 32k-token conversation?")
print(result.trace())

## 4. When things go wrong

Wrong tool names, bad arguments and crashing tools all come back to the model as error messages, so it can recover instead of the program crashing.

In [ ]:
messy = ScriptedModel([
    ModelResponse(tool_calls=[ToolCall("calc", {"expression": "2+2"})]),
    ModelResponse(tool_calls=[ToolCall("calculator", {"expression": 4})]),
    ModelResponse(tool_calls=[ToolCall("calculator", {"expression": "10 / 0"})]),
    ModelResponse(tool_calls=[ToolCall("calculator", {"expression": "2 + 2"})]),
    ModelResponse(text="2 + 2 = 4"),
])
print(run_agent(messy, tools, "What is 2 + 2?").trace())

## 5. Guardrails: approval and step limits

Tools with real-world effects should need a human's OK. And every loop needs a limit, because a model can get stuck repeating itself.

In [ ]:
outbox = []
send_email = Tool("send_email", "Send an email.",
                  {"type": "object", "properties": {"to": {"type": "string"}, "body": {"type": "string"}}, "required": ["to", "body"]},
                  fn=lambda to, body: outbox.append(to) or f"sent to {to}", requires_approval=True)
wants_to_send = [ModelResponse(tool_calls=[ToolCall("send_email", {"to": "boss@example.com", "body": "I quit"})]),
                 ModelResponse(text="I didn't send it: you didn't approve the email.")]

print(run_agent(ScriptedModel(list(wants_to_send)), [send_email], "Email my boss", approve=lambda call: False).trace())
print("outbox:", outbox)

stuck = ScriptedModel([ModelResponse(tool_calls=[ToolCall("calculator", {"expression": "1 + 1"})])] * 50)
r = run_agent(stuck, tools, "q", max_steps=5)
print(f"\nstuck model: stopped after {r.steps} steps because {r.stopped_because}")

## 6. Why long agent runs get expensive

Every step re-sends the whole conversation so far. Without caching, the tokens processed grow with the *square* of the number of steps.

In [ ]:
base, per_step = 10_000, 1_000           # system prompt + tools + question; each call and result adds ~1k
for steps in [1, 5, 10, 20, 50]:
    plain = tokens_processed(base, per_step, steps, cached=False)
    cached = tokens_processed(base, per_step, steps, cached=True)
    print(f"{steps:>3} steps: {plain:>9,} tokens without caching, {cached:>7,} with ({plain / cached:4.1f}x)")

## Try it

- Add a tool of your own (for example, one that returns today's date) and a question that needs it.
- Rewrite `search_docs`'s description to be vague ("Does stuff"). Why would a real model struggle to pick it?
- Make `approve` print the call and ask for input with `input()`. That's a basic human-in-the-loop.
- In section 6, how many steps until a run processes over a million tokens without caching?